# Editor con vista 3D

`test_project_editor.ipynb` prueba el editor con un proyecto pequeño, que no
tiene geometría. Este prueba la parte 3D, y para eso hace falta un edificio de
verdad: el del curso HULC, 150 componentes y 74 superficies con geometría.

**Antes de empezar**: selecciona el kernel del `.venv` del proyecto (en VS Code,
arriba a la derecha → *Select Kernel* → *Python Environments* → `.venv`).

El visor descarga `plotly.js` de esm.sh la primera vez que lo abres, así que
hace falta conexión. Son 1,7 MB y **solo se descargan al pulsar "Show 3D"**.

## 1. La geometría como dato

In [4]:
import opensimula as osm

sim = osm.Simulation()
sim.console_print = False
pro = sim.new_project("hulc")
pro.read_json("edificio_curso_hulc.json")

geo = pro.geometry_dict()
print("mallas:", len(geo["meshes"]))
print("espacios:", geo["spaces"])

mallas: 74
espacios: ['P01_E01', 'P02_E01', 'P02_E02', 'P02_E03', 'P02_E04', 'P02_E05', 'P02_E06', 'P03_E01']


In [5]:
# Cada malla sabe de qué componente viene y qué espacios limita.
# Eso es lo que permitirá filtrar la vista por espacio.
for m in geo["meshes"][:3]:
    print(f'{m["component"]:18} {m["component_type"]:16} {m["surface_type"]:10} {m["spaces"]}')

interiores = [m for m in geo["meshes"] if m["surface_type"] == "INTERIOR"]
print(f'\nparticiones interiores: {len(interiores)} (limitan dos espacios)')
print("ejemplo:", interiores[0]["component"], interiores[0]["spaces"])

P01_E01_PCT001     Building_surface EXTERIOR   ['P01_E01']
P01_E01_PCT002     Building_surface EXTERIOR   ['P01_E01']
P01_E01_PCT003     Building_surface EXTERIOR   ['P01_E01']

particiones interiores: 21 (limitan dos espacios)
ejemplo: P02_E01_Med002 ['P02_E01', 'P02_E02']


## 2. El editor con el visor

In [6]:
editor = pro.editor()
editor

### Qué mirar

Pulsa **Show 3D** abajo a la derecha. El widget crece y aparece el visor bajo el
formulario.

1. Que salga el edificio **entero**, no un trozo. La cámara se calcula a partir
   de las dimensiones del modelo; la de plotly por defecto lo recortaría.
2. Pasa el ratón por una superficie: debe salir su nombre.
3. Gira y acerca con el ratón. Los huecos se ven en azul, las particiones
   interiores en verde.
4. Pincha una superficie en la lista de la izquierda y comprueba que el
   formulario sigue respondiendo con el visor abierto.
5. Vuelve a pulsar el botón: se oculta y el widget recupera su altura.

### El filtro por espacio

Ese es el problema que resuelve la barra del visor: **desde fuera, los muros
exteriores tapan todos los cerramientos interiores**. En el desplegable
*Space* elige `P02_E01`.

1. Solo quedan las superficies de ese espacio. El contador de la derecha lo
   dice: `8 of 74`.
2. Aparecen sus **particiones interiores en verde**, que antes estaban ocultas.
3. El local sigue **en su sitio y a su tamaño**: los ejes están fijados al
   edificio completo, no a lo que deja el filtro, así que la cámara no salta ni
   el espacio se infla para llenar el panel.
4. La casilla *Openings* quita los huecos, útil cuando estorban para ver el
   cerramiento que hay detrás.

### Selección enlazada

Va en los dos sentidos:

- Pincha una superficie **en la lista** de la izquierda: se pone naranja en el 3D,
  con el contorno más grueso. Es la forma de localizar un cerramiento concreto
  entre setenta y cuatro.
- Pincha una superficie **en el 3D**: la lista salta a ella y el formulario
  muestra sus parámetros.

Combínalo con el filtro: aísla un espacio y ve pinchando sus particiones para
revisarlas una a una.

## 3. Qué actualiza el visor y cuándo

La geometría no sale del JSON: hace falta el **modelo construido**, con las
referencias resueltas y los orígenes situados respecto al edificio. Así que el
visor enseña el **proyecto**, no el documento que estás editando.

Lo que cambia es cuándo se actualiza el proyecto:

- **Mover un vértice o cambiar una altura** es un cambio de valor: llega al
  proyecto solo, y el visor se refresca sin que pulses nada.
- **Añadir, borrar o renombrar** espera a **Apply**, y hasta entonces el visor
  sigue mostrando la geometría anterior.

In [4]:
# Una superficie exterior cualquiera
sup = pro.component("P02_E01_PE001")
print("shape :", sup.parameter("shape").value)
print("height:", sup.parameter("height").value, "m")

shape : RECTANGLE
height: 3.0 m


In [5]:
# Cambiar su altura: es un valor, así que entra solo
doc = {**editor.value, "components": [dict(c) for c in editor.value["components"]]}
comp = next(c for c in doc["components"] if c["name"] == "P02_E01_PE001")
comp["height"] = comp["height"] * 2
editor.value = doc

print("documento:", comp["height"], "m")
print("proyecto :", pro.component("P02_E01_PE001").parameter("height").value, "m   <- ya aplicado")
print("pending  :", editor.pending)

documento: 6 m
proyecto : 6.0 m   <- ya aplicado
pending  : []


Mira el visor de arriba: esa fachada ha crecido, **sin que la vista vuelva a
su ángulo inicial**. Gira el modelo antes de ejecutar la celda siguiente para
comprobarlo: la cámara se queda donde la dejes.

In [6]:
# Y de vuelta. Copia nueva, no mutar la que ya tiene el widget: si se muta en
# su sitio, traitlets compara y no ve cambio, así que no se entera de nada.
doc = {**editor.value, "components": [dict(c) for c in editor.value["components"]]}
comp = next(c for c in doc["components"] if c["name"] == "P02_E01_PE001")
comp["height"] = comp["height"] / 2
editor.value = doc

print("altura restaurada:", pro.component("P02_E01_PE001").parameter("height").value, "m")

altura restaurada: 1.5 m


### Lo estructural sí espera

Borrar una superficie cambia la geometría, pero obliga a reconstruir el
proyecto, así que el visor no se entera hasta que pulses **Apply**.

In [ ]:
doc = {**editor.value, "components": [c for c in editor.value["components"]
                                       if c["name"] != "P02_E01_PE001"]}
editor.value = doc
print("pending          :", editor.pending)
print("mallas en el visor:", len(editor.geometry["meshes"]), "  <- todavía la de antes")

editor.apply()
print("tras apply       :", len(editor.geometry["meshes"]), "mallas, pending", editor.pending)

## 4. Recargar la vista a mano

`refresh_geometry()` vuelve a leer la geometría del proyecto. `apply()` ya lo
hace, pero es útil si has modificado el proyecto por código.

In [ ]:
# Una superficie cualquiera de las que queden: la anterior la borramos arriba.
otra = pro.component_list("Building_surface")[0]
otra.parameter("azimuth").value = 45
editor.refresh_geometry()   # sin esto, el visor seguiría con la orientación vieja

print(otra.parameter("name").value, "-> azimuth", otra.parameter("azimuth").value)

## 5. La vista de siempre, aparte

`show_3D()` sigue funcionando igual. Para componer o guardar la figura en vez de
solo mirarla, `plotly_figure_3D()` da el mismo `go.Figure` sin mostrarlo.

In [ ]:
fig = pro.plotly_figure_3D()
print(type(fig).__name__, "con", len(fig.data), "trazas")